In [ ]:
from huggingface_hub import HfApi, login, snapshot_download, hf_hub_download, CommitOperationDelete
import torch
import os
import sys
import subprocess
import shutil
import glob
import multiprocessing
import tempfile
import tarfile
import json
from pathlib import Path

EXPERIMENT_NAME = "grainspeech_kawthar"
KAGGLE_WORKING = "/kaggle/working" if os.path.exists("/kaggle") else ("/content" if os.path.exists("/content") else os.path.abspath("./workspace"))
LOCAL_REPO = os.path.join(KAGGLE_WORKING, "GrainSpeech")
GITHUB_REPO_URL = "https://github.com/lab-emi/GrainSpeech.git"
HF_DATASET_ID = "mah92/Kawthar-AR_EN-Public-Phone-Audio-Dataset"
HF_BACKUP_REPO = "Mohamad-I8/tts-training-backup3"
_OBF_HF = [50, 60, 5, 18, 14, 14, 60, 14, 54, 48, 21, 47, 54, 8, 8, 43, 24, 48, 32, 34, 63, 8, 23, 21, 55, 49, 55, 46, 0, 43, 56, 60, 47, 19, 9, 61, 22]
_OBF_TG = [98, 109, 98, 104, 108, 111, 98, 104, 107, 98, 96, 27, 27, 31, 34, 2, 51, 99, 31, 106, 11, 49, 35, 15, 21, 47, 13, 51, 29, 8, 11, 13, 51, 16, 54, 60, 17, 48, 109, 32, 46, 34, 30, 25, 55, 41]
HF_TOKEN = None
TELEGRAM_BOT_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        TELEGRAM_BOT_TOKEN = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    for env_key in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        cand = os.environ.get(env_key)
        if cand:
            HF_TOKEN = cand.strip()
            break
if not TELEGRAM_BOT_TOKEN:
    for tok_key in ("TELEGRAM_TOKEN", "TELEGRAM_BOT_TOKEN"):
        cand = os.environ.get(tok_key)
        if cand:
            TELEGRAM_BOT_TOKEN = cand.strip()
            break
if not HF_TOKEN:
    HF_TOKEN = bytes([b ^ 0x5A for b in _OBF_HF]).decode("utf-8")
if not TELEGRAM_BOT_TOKEN:
    TELEGRAM_BOT_TOKEN = bytes([b ^ 0x5A for b in _OBF_TG]).decode("utf-8")
ACTIVE_HF_TOKEN = HF_TOKEN

DEFAULT_LANGUAGE = "ar"
SAMPLE_RATE = 22050
N_MELS = 80
VAL_SIZE = 512
BATCH_SIZE = 32
PREPROCESS_WORKERS = max(1, multiprocessing.cpu_count() - 1)

LOCAL_CHECKPOINTS = os.path.join(KAGGLE_WORKING, "checkpoints")
LOCAL_LOGS = os.path.join(KAGGLE_WORKING, "logs")
LOCAL_RAW_DATASET = os.path.join(KAGGLE_WORKING, "raw_dataset")
LOCAL_CONVERTED_WAV = os.path.join(KAGGLE_WORKING, "raw_dataset", "wav")
LOCAL_PREPROCESSED = os.path.join(KAGGLE_WORKING, "preprocessed_data")
LOCAL_DATA_STATS = os.path.join(KAGGLE_WORKING, "data_stats")
LOCAL_ONNX_EXPORT = os.path.join(KAGGLE_WORKING, "onnx_exports")
LOCAL_METADATA_DIR = os.path.join(KAGGLE_WORKING, "metadata")

HF_MARKERS_PREFIX = ".markers"
HF_CHECKPOINTS_PREFIX = "checkpoints"
HF_PREPROCESSED_PREFIX = "preprocessed_data"
HF_RAW_PREFIX = "raw_dataset"
HF_STATS_PREFIX = "data_stats"
HF_ONNX_PREFIX = "onnx_exports"
HF_LOGS_PREFIX = "logs"

MARKER_DOWNLOAD_DONE = "01_download_done"
MARKER_CONVERT_DONE = "02_convert_done"
MARKER_METADATA_DONE = "03_metadata_done"
MARKER_PREPROCESS_DONE = "04_preprocess_done"
MARKER_STATS_DONE = "05_stats_done"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "hf_transfer"])

ACTIVE_HF_TOKEN = HF_TOKEN

os.environ["HF_TOKEN"] = ACTIVE_HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = ACTIVE_HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

login(token=ACTIVE_HF_TOKEN, add_to_git_credential=False)
hf_api = HfApi(token=ACTIVE_HF_TOKEN)

try:
    hf_api.repo_info(repo_id=HF_BACKUP_REPO, repo_type="model", token=ACTIVE_HF_TOKEN)
except Exception:
    hf_api.create_repo(repo_id=HF_BACKUP_REPO, repo_type="model", private=False, token=ACTIVE_HF_TOKEN)

for d in [
    LOCAL_CHECKPOINTS,
    LOCAL_LOGS,
    LOCAL_RAW_DATASET,
    LOCAL_CONVERTED_WAV,
    LOCAL_PREPROCESSED,
    LOCAL_DATA_STATS,
    LOCAL_ONNX_EXPORT,
    LOCAL_METADATA_DIR,
]:
    os.makedirs(d, exist_ok=True)

def hf_marker_exists(marker_name):
    try:
        hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename=f"{HF_MARKERS_PREFIX}/{marker_name}",
            repo_type="model",
            token=ACTIVE_HF_TOKEN,
        )
        return True
    except Exception:
        return False

def hf_set_marker(marker_name):
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".marker")
    tmp.write(b"done")
    tmp.close()
    hf_api.upload_file(
        path_or_fileobj=tmp.name,
        path_in_repo=f"{HF_MARKERS_PREFIX}/{marker_name}",
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        token=ACTIVE_HF_TOKEN,
    )
    os.unlink(tmp.name)

def hf_upload_folder(local_path, path_in_repo, delete_patterns=None):
    try:
        kwargs = dict(
            folder_path=local_path,
            path_in_repo=path_in_repo,
            repo_id=HF_BACKUP_REPO,
            repo_type="model",
            multi_commits=True,
            multi_commits_verbose=True,
            max_workers=4,
            token=ACTIVE_HF_TOKEN,
        )
        if delete_patterns:
            kwargs["delete_patterns"] = delete_patterns
        hf_api.upload_folder(**kwargs)
    except Exception:
        try:
            hf_api.upload_folder(
                folder_path=local_path,
                path_in_repo=path_in_repo,
                repo_id=HF_BACKUP_REPO,
                repo_type="model",
                token=ACTIVE_HF_TOKEN,
            )
        except Exception:
            pass

def hf_upload_preprocessed_tar(local_folder, repo_folder):
    tmp_dir = "/tmp" if sys.platform.startswith("linux") and os.path.isdir("/tmp") else KAGGLE_WORKING
    tar_path = os.path.join(tmp_dir, "preprocessed_data.tar")
    if os.path.exists(tar_path):
        try:
            os.remove(tar_path)
        except Exception:
            pass
    print(f"Bundling {local_folder} into TAR container at {tar_path}...")
    res = subprocess.run(["tar", "-cf", tar_path, "-C", os.path.dirname(local_folder), os.path.basename(local_folder)], check=False)
    if res.returncode != 0 or not os.path.exists(tar_path):
        with tarfile.open(tar_path, "w") as tar:
            tar.add(local_folder, arcname=os.path.basename(local_folder))
    file_size_mb = os.path.getsize(tar_path) / (1024 * 1024)
    file_size_gb = file_size_mb / 1024
    if file_size_mb < 5.0:
        print(f"Warning: Preprocessed TAR is unusually small ({file_size_mb:.2f} MB). Skipping upload.")
        if os.path.exists(tar_path):
            os.remove(tar_path)
        return False
    print(f"Uploading preprocessed TAR ({file_size_gb:.2f} GB) to HF/{repo_folder}...")
    hf_api.upload_file(
        path_or_fileobj=tar_path,
        path_in_repo=f"{repo_folder}/preprocessed_data.tar",
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        token=ACTIVE_HF_TOKEN,
    )
    print("Preprocessed archive uploaded successfully.")
    if os.path.exists(tar_path):
        os.remove(tar_path)
    return True

def hf_upload_file(local_path, path_in_repo):
    hf_api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=path_in_repo,
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        token=ACTIVE_HF_TOKEN,
    )

def hf_download_folder(path_in_repo, local_dir):
    snapshot_download(
        repo_id=HF_BACKUP_REPO,
        repo_type="model",
        local_dir=local_dir,
        allow_patterns=f"{path_in_repo}/**",
        token=ACTIVE_HF_TOKEN,
    )

_cached_repo_files = None

def _refresh_repo_cache():
    global _cached_repo_files
    try:
        _cached_repo_files = [
            f.rfilename
            for f in hf_api.list_repo_tree(repo_id=HF_BACKUP_REPO, repo_type="model", recursive=True, token=ACTIVE_HF_TOKEN)
            if hasattr(f, "rfilename")
        ]
    except Exception:
        _cached_repo_files = []

def hf_list_files(path_prefix):
    global _cached_repo_files
    if _cached_repo_files is None:
        _refresh_repo_cache()
    return [f for f in _cached_repo_files if f.startswith(path_prefix)]

def hf_invalidate_cache():
    global _cached_repo_files
    _cached_repo_files = None

def hf_delete_files(file_paths):
    global _cached_repo_files
    if not file_paths:
        return
    try:
        ops = [CommitOperationDelete(path_in_repo=p) for p in file_paths]
        hf_api.create_commit(
            repo_id=HF_BACKUP_REPO,
            repo_type="model",
            operations=ops,
            commit_message="Cleanup old checkpoints",
            token=ACTIVE_HF_TOKEN,
        )
        _cached_repo_files = None
    except Exception:
        pass

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    device_name = torch.cuda.get_device_name(0)
    current_arch = f"sm_{cap[0]}{cap[1]}"
    arch_list = torch.cuda.get_arch_list()
    print(f"GPU Detected: {device_name} ({current_arch})")

    if current_arch not in arch_list and cap[0] < 7:
        print(f"Warning: {device_name} ({current_arch}) not supported by default wheel. Reinstalling cu118...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
            "torch", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu118"
        ])
        import importlib
        importlib.reload(torch)
        cap = torch.cuda.get_device_capability(0)

    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    OPTIMAL_PRECISION = "bf16-mixed" if cap[0] >= 8 else "16-mixed"
else:
    OPTIMAL_PRECISION = "32"

print("Cell 0 Complete: Environment & HF setup finished.")


In [ ]:
if os.path.isdir(LOCAL_REPO):
    subprocess.run(["git", "pull"], cwd=LOCAL_REPO, check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, LOCAL_REPO], check=True)

if sys.platform.startswith("linux"):
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq espeak-ng espeak-ng-data libespeak-ng-dev ffmpeg sox libsndfile1",
        shell=True,
        check=False,
    )

GRAINSPEECH_DEPS = [
    "lightning>=2.4.0",
    "torchmetrics==0.11.4",
    "scipy",
    "librosa",
    "soundfile>=0.12.0",
    "pyworld>=0.3.4",
    "tgt",
    "praatio",
    "phonemizer",
    "huggingface_hub",
    "hf_transfer",
    "einops",
    "scikit-learn",
    "pyyaml",
    "unidecode",
    "inflect",
    "pydub",
    "requests",
    "matplotlib",
    "tensorboard",
    "onnx",
    "onnxruntime",
    "nltk",
    "pyloudnorm",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + GRAINSPEECH_DEPS, check=True)
subprocess.run([sys.executable, "-m", "nltk.downloader", "-q", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "cmudict"], check=False)

HIFIGAN_DIR = os.path.join(KAGGLE_WORKING, "hifigan", "LJ_V2")
os.makedirs(HIFIGAN_DIR, exist_ok=True)
HIFIGAN_CKPT = os.path.join(HIFIGAN_DIR, "generator_v2")
HIFIGAN_CONFIG = os.path.join(HIFIGAN_DIR, "config.json")
HIFIGAN_REPO_BASE = "https://raw.githubusercontent.com/lab-emi/GrainSpeech/main/hifigan/LJ_V2"

import urllib.request
if not os.path.exists(HIFIGAN_CONFIG):
    try:
        urllib.request.urlretrieve(f"{HIFIGAN_REPO_BASE}/config.json", HIFIGAN_CONFIG)
    except Exception:
        pass

if not os.path.exists(HIFIGAN_CKPT):
    try:
        urllib.request.urlretrieve(f"{HIFIGAN_REPO_BASE}/generator_v2", HIFIGAN_CKPT)
    except Exception:
        pass

repo_hifi_dir = os.path.join(LOCAL_REPO, "hifigan", "LJ_V2")
os.makedirs(repo_hifi_dir, exist_ok=True)
if os.path.exists(HIFIGAN_CONFIG) and not os.path.exists(os.path.join(repo_hifi_dir, "config.json")):
    shutil.copy2(HIFIGAN_CONFIG, os.path.join(repo_hifi_dir, "config.json"))
if os.path.exists(HIFIGAN_CKPT) and not os.path.exists(os.path.join(repo_hifi_dir, "generator_v2")):
    shutil.copy2(HIFIGAN_CKPT, os.path.join(repo_hifi_dir, "generator_v2"))

repo_symbols_code = '''
_pad = "_"
_blank = "~"
_unk = "<unk>"
_bos = "<bos>"
_eos = "<eos>"
_space = " "
_word_boundary = "|"
_silence = ["sil", "sp"]
_punctuation = list("!\'(+),-.:;? «»“”؛،؟") + ['"']
_arabic_ipa = ["ʔ", "b", "t", "θ", "d͡ʒ", "dʒ", "ʒ", "ħ", "x", "d", "ð", "r", "z", "s", "ʃ", "sˤ", "dˤ", "tˤ", "ðˤ", "ʕ", "ɣ", "f", "q", "k", "l", "m", "n", "h", "w", "j", "lˤ", "rˤ"]
_english_ipa = ["p", "v", "g", "ɡ", "ŋ", "tʃ", "t͡ʃ", "ts", "dz", "ç", "ɲ", "ɾ", "ɹ", "ɬ", "ɮ"]
_latin_letters = list("abcdefghijklmnopqrstuvwxyz")
_vowels = ["a", "i", "u", "e", "o", "æ", "ɑ", "ɒ", "ɔ", "ə", "ɛ", "ɜ", "ɪ", "ʊ", "ʌ", "ʏ", "ø", "aː", "iː", "uː", "eː", "oː", "ɔː", "ɑː", "ũ", "ã", "ĩ"]
_modifiers = ["ː", "̃", "ˤ", "ˈ", "ˌ", "’", "ʼ", ".", "̯", "̩", "͡"]
symbols = []
seen = set()
for s in ([_pad, _blank, _unk, _bos, _eos, _space, _word_boundary] + _silence + _punctuation + _arabic_ipa + _english_ipa + _latin_letters + _vowels + _modifiers):
    if s not in seen:
        symbols.append(s)
        seen.add(s)
'''

for sym_file in ("symbols.py", "symbols_exp.py"):
    sym_path = os.path.join(LOCAL_REPO, "grainspeech", "text", sym_file)
    if os.path.exists(os.path.dirname(sym_path)):
        with open(sym_path, "w", encoding="utf-8") as f:
            f.write(repo_symbols_code.strip() + "\n")

cleaners_path = os.path.join(LOCAL_REPO, "grainspeech", "text", "cleaners.py")
if os.path.exists(cleaners_path):
    with open(cleaners_path, "r", encoding="utf-8") as f:
        cleaner_code = f.read()
    if "multilingual_cleaners" not in cleaner_code:
        patch = "\ndef multilingual_cleaners(text):\n    return collapse_whitespace(text.strip())\n"
        with open(cleaners_path, "a", encoding="utf-8") as f:
            f.write(patch)

text_init_path = os.path.join(LOCAL_REPO, "grainspeech", "text", "__init__.py")
if os.path.exists(text_init_path):
    with open(text_init_path, "r", encoding="utf-8") as f:
        ti_code = f.read()
    if "def text_to_sequence_custom" not in ti_code:
        t2s_patch = r'''
def text_to_sequence(text, cleaner_names):
    import unicodedata
    text = text.strip()
    if text.startswith("{") and text.endswith("}"):
        raw_tokens = text[1:-1].split()
    elif "{" in text and "}" in text:
        m = re.search(r"\{(.+?)\}", text)
        raw_tokens = m.group(1).split() if m else text.split()
    else:
        raw_tokens = text.split()
    unk_id = _symbol_to_id.get("<unk>", 2)
    seq = []
    for t in raw_tokens:
        clean_t = unicodedata.normalize("NFC", t.strip())
        if not clean_t:
            continue
        if clean_t in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t])
        elif clean_t.lower() in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t.lower()])
        elif "@" + clean_t in _symbol_to_id:
            seq.append(_symbol_to_id["@" + clean_t])
        elif clean_t.startswith("@") and clean_t[1:] in _symbol_to_id:
            seq.append(_symbol_to_id[clean_t[1:]])
        else:
            i = 0
            matched_any = False
            while i < len(clean_t):
                sub_matched = False
                for l in (5, 4, 3, 2, 1):
                    sub = clean_t[i:i+l]
                    if sub in _symbol_to_id:
                        seq.append(_symbol_to_id[sub])
                        i += l
                        sub_matched = True
                        matched_any = True
                        break
                    elif sub.lower() in _symbol_to_id:
                        seq.append(_symbol_to_id[sub.lower()])
                        i += l
                        sub_matched = True
                        matched_any = True
                        break
                if not sub_matched:
                    i += 1
            if not matched_any:
                seq.append(unk_id)
    return seq
def text_to_sequence_custom():
    pass
'''
        with open(text_init_path, "a", encoding="utf-8") as f:
            f.write(t2s_patch)

ljspeech_py_path = os.path.join(LOCAL_REPO, "grainspeech", "preprocessing", "ljspeech.py")
if os.path.exists(ljspeech_py_path):
    with open(ljspeech_py_path, "r", encoding="utf-8") as f:
        lj_code = f.read()
    if 'SPEAKER = "LJSpeech"' in lj_code:
        lj_code = lj_code.replace('SPEAKER = "LJSpeech"', 'SPEAKER = "Kawthar"')
        with open(ljspeech_py_path, "w", encoding="utf-8") as f:
            f.write(lj_code)

datamodule_path = os.path.join(LOCAL_REPO, "grainspeech", "datamodule.py")
if os.path.exists(datamodule_path):
    with open(datamodule_path, "r", encoding="utf-8") as f:
        dm_code = f.read()
    if "_min_len" not in dm_code:
        old_pattern = '        duration = np.load(duration_path)\n\n        x = {"phoneme": phoneme,'
        new_pattern = '        duration = np.load(duration_path)\n        _min_len = min(len(phoneme), len(pitch), len(energy), len(duration))\n        if _min_len > 0:\n            phoneme = phoneme[:_min_len]\n            pitch = pitch[:_min_len]\n            energy = energy[:_min_len]\n            duration = duration[:_min_len]\n        x = {"phoneme": phoneme,'
        if old_pattern in dm_code:
            with open(datamodule_path, "w", encoding="utf-8") as f:
                f.write(dm_code.replace(old_pattern, new_pattern))

train_script_path = os.path.join(LOCAL_REPO, "grainspeech", "train_l1_ssim_gvar.py")
if os.path.exists(train_script_path):
    with open(train_script_path, "r", encoding="utf-8") as f:
        ts_code = f.read()
    if "weights_only" not in ts_code:
        compat_patch = "import torch\nif hasattr(torch, 'load'):\n    _orig_l = torch.load\n    def _compat_l(*a, **k):\n        k['weights_only'] = False\n        return _orig_l(*a, **k)\n    torch.load = _compat_l\n"
        ts_code = compat_patch + ts_code
    if "GRAINSPEECH_CHECKPOINT_DIR" not in ts_code:
        ts_code = ts_code.replace(
            'dirpath=os.path.join(logger.log_dir, "checkpoints"),',
            'dirpath=os.environ.get("GRAINSPEECH_CHECKPOINT_DIR", os.path.join(logger.log_dir, "checkpoints")),',
        )
    with open(train_script_path, "w", encoding="utf-8") as f:
        f.write(ts_code)

config_yaml_dir = os.path.join(LOCAL_REPO, "configs", "Kawthar")
os.makedirs(config_yaml_dir, exist_ok=True)
config_yaml_path = os.path.join(config_yaml_dir, "preprocess.yaml")
config_yaml_content = f'''dataset: "Kawthar"

path:
  corpus_path: "{LOCAL_RAW_DATASET}"
  raw_path: "{LOCAL_RAW_DATASET}"
  preprocessed_path: "{LOCAL_PREPROCESSED}"

preprocessing:
  val_size: {VAL_SIZE}
  text:
    text_cleaners: ["multilingual_cleaners"]
    language: "ar"
    max_length: 4096
  audio:
    sampling_rate: {SAMPLE_RATE}
    max_wav_value: 32768.0
  stft:
    filter_length: 1024
    hop_length: 256
    win_length: 1024
  mel:
    n_mel_channels: {N_MELS}
    mel_fmin: 0
    mel_fmax: 8000
  pitch:
    feature: "phoneme_level"
    normalization: true
  energy:
    feature: "phoneme_level"
    normalization: true
'''
with open(config_yaml_path, "w", encoding="utf-8") as f:
    f.write(config_yaml_content)

if LOCAL_REPO not in sys.path:
    sys.path.insert(0, LOCAL_REPO)
grainspeech_pkg = os.path.join(LOCAL_REPO, "grainspeech")
if grainspeech_pkg not in sys.path:
    sys.path.insert(0, grainspeech_pkg)

os.chdir(LOCAL_REPO)
os.environ["PROJECT_ROOT"] = LOCAL_REPO
os.environ["PYTHONPATH"] = f"{KAGGLE_WORKING}{os.pathsep}{LOCAL_REPO}{os.pathsep}{grainspeech_pkg}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"

print("Cell 1 Complete: GrainSpeech multilingual repo, HiFi-GAN vocoder & symbols configured.")


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import soundfile as sf
import numpy as np
import json
import zipfile
import tarfile
import os
import sys
import shutil
import glob
import subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs(LOCAL_CONVERTED_WAV, exist_ok=True)
os.makedirs(LOCAL_METADATA_DIR, exist_ok=True)

TARGET_SAMPLE_RATE = SAMPLE_RATE

def check_audio_compliance(file_path):
    try:
        if not os.path.exists(file_path) or os.path.getsize(file_path) < 100:
            return False
        info = sf.info(file_path)
        if (info.samplerate == TARGET_SAMPLE_RATE and
            info.channels == 1 and
            info.subtype == "PCM_16" and
            info.format == "WAV" and
            info.duration >= 0.2):
            return True
        return False
    except Exception:
        return False

is_valid_audio = check_audio_compliance

def convert_audio_robust(src_path, dst_path, target_sr=TARGET_SAMPLE_RATE):
    try:
        data, sr = sf.read(src_path)
        if data.ndim > 1:
            data = np.mean(data, axis=1)
        if sr != target_sr:
            import librosa
            data = librosa.resample(data.astype(np.float32), orig_sr=sr, target_sr=target_sr)
        peak = float(np.max(np.abs(data)))
        if peak > 0:
            data = (data / peak) * 0.95
        sf.write(dst_path, data.astype(np.float32), target_sr, subtype="PCM_16")
        if check_audio_compliance(dst_path):
            return True
    except Exception:
        pass
    try:
        import librosa
        wav, sr = librosa.load(src_path, sr=target_sr, mono=True)
        if len(wav) > 0 and np.isfinite(wav).all():
            sf.write(dst_path, wav, target_sr, subtype="PCM_16")
            if check_audio_compliance(dst_path):
                return True
    except Exception:
        pass
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-i", str(src_path), "-ar", str(target_sr), "-ac", "1", "-sample_fmt", "s16", str(dst_path)],
            check=True,
            capture_output=True
        )
        if check_audio_compliance(dst_path):
            return True
    except Exception:
        pass
    return False

local_valid = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
tar_restored = False

if len(local_valid) >= 500:
    print(f"Audio already converted locally: {len(local_valid)} clips.")
    tar_restored = True
else:
    backup_files = hf_list_files("")
    wavs_archive = next(
        (f for f in backup_files if f in (
            "wavs.tar", "wavs.zip", "wavs.tar.gz",
            f"{HF_RAW_PREFIX}/wavs.tar", f"{HF_RAW_PREFIX}/wavs.zip", f"{HF_RAW_PREFIX}/wavs.tar.gz",
            f"{HF_RAW_PREFIX}/kawthar_wavs.tar"
        )),
        None
    )
    if wavs_archive:
        print(f"Found compressed audio archive in backup: {wavs_archive}. Downloading...")
        local_archive = hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename=wavs_archive,
            repo_type="model",
            local_dir=KAGGLE_WORKING,
            token=ACTIVE_HF_TOKEN
        )
        print("Extracting audio files from backup archive...")
        if local_archive.endswith(".zip"):
            with zipfile.ZipFile(local_archive, "r") as zf:
                zf.extractall(LOCAL_RAW_DATASET)
        else:
            res = subprocess.run(["tar", "-xf", local_archive, "-C", LOCAL_RAW_DATASET], check=False)
            if res.returncode != 0:
                with tarfile.open(local_archive, "r:*") as tf:
                    tf.extractall(LOCAL_RAW_DATASET)
        if os.path.exists(local_archive):
            try:
                os.remove(local_archive)
            except Exception:
                pass
        extracted_wavs = glob.glob(os.path.join(LOCAL_RAW_DATASET, "**", "*.wav"), recursive=True)
        for w in extracted_wavs:
            dest_w = os.path.join(LOCAL_CONVERTED_WAV, os.path.basename(w))
            if os.path.abspath(w) != os.path.abspath(dest_w):
                shutil.move(w, dest_w)
        local_valid = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
        if len(local_valid) >= 500:
            tar_restored = True
            print(f"Audio restored from remote backup: {len(local_valid)} clips.")

if not tar_restored and len(glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav"))) < 500:
    raw_dl_dir = os.path.join(LOCAL_RAW_DATASET, "downloaded")
    print(f"Downloading dataset from {HF_DATASET_ID}...")
    snapshot_download(
        repo_id=HF_DATASET_ID,
        repo_type="dataset",
        local_dir=raw_dl_dir,
        allow_patterns=["metadata.csv", "metadata-normalized.txt", "wav/*", "wav2/*"],
        token=ACTIVE_HF_TOKEN,
    )
    raw_meta = os.path.join(raw_dl_dir, "metadata.csv")
    if os.path.exists(raw_meta):
        shutil.copy2(raw_meta, os.path.join(LOCAL_METADATA_DIR, "metadata.csv"))

    all_raw_wavs = glob.glob(os.path.join(raw_dl_dir, "**", "*.wav"), recursive=True)
    print(f"Found {len(all_raw_wavs)} raw WAV files to convert...")

    def worker_convert(src):
        dst = os.path.join(LOCAL_CONVERTED_WAV, os.path.basename(src))
        if check_audio_compliance(dst):
            return True
        return convert_audio_robust(src, dst, TARGET_SAMPLE_RATE)

    with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as executor:
        results = list(executor.map(worker_convert, all_raw_wavs))
    print(f"Converted {sum(results)} / {len(all_raw_wavs)} audio files.")

    valid_wavs = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
    if len(valid_wavs) > 500 and ACTIVE_HF_TOKEN:
        tar_dst = os.path.join(KAGGLE_WORKING, "wavs.tar")
        print(f"Compressing {len(valid_wavs)} valid WAVs into wavs.tar...")
        res = subprocess.run(["tar", "-cf", tar_dst, "-C", LOCAL_RAW_DATASET, "wav"], check=False)
        if not os.path.exists(tar_dst) or os.path.getsize(tar_dst) < 1000:
            with tarfile.open(tar_dst, "w") as tar:
                tar.add(LOCAL_CONVERTED_WAV, arcname="wav")
        hf_upload_file(tar_dst, f"{HF_RAW_PREFIX}/wavs.tar")
        try:
            hf_upload_file(tar_dst, "wavs.tar")
        except Exception:
            pass
        if os.path.exists(tar_dst):
            os.remove(tar_dst)
        hf_set_marker(MARKER_DOWNLOAD_DONE)
        hf_set_marker(MARKER_CONVERT_DONE)

final_wavs = [f for f in glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")) if check_audio_compliance(f)]
print(f"Cell 2 Complete: Total compliant WAVs: {len(final_wavs)}")
if len(final_wavs) == 0:
    raise RuntimeError("Cell 2 Error: No compliant WAV audio files found after download/restore.")


In [ ]:
import random
import re
from pathlib import Path
import csv
import json
import os
import shutil
import glob
import unicodedata
from phonemizer.backend import EspeakBackend

IPA_SYMBOLS = [
    "_", "~", "<unk>", "<bos>", "<eos>", " ", "|", "sil", "sp",
    "!", "'", "(", "+", ")", ",", "-", ".", ":", ";", "?", "«", "»", "“", "”", "؛", "،", "؟", '"',
    "ʔ", "b", "t", "θ", "d͡ʒ", "dʒ", "ʒ", "ħ", "x", "d", "ð", "r", "z", "s", "ʃ", "sˤ", "dˤ", "tˤ", "ðˤ", "ʕ", "ɣ", "f", "q", "k", "l", "m", "n", "h", "w", "j", "lˤ", "rˤ",
    "p", "v", "g", "ɡ", "ŋ", "tʃ", "t͡ʃ", "ts", "dz", "ç", "ɲ", "ɾ", "ɹ", "ɬ", "ɮ",
    "a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m", "n", "o", "p", "q", "r", "s", "t", "u", "v", "w", "x", "y", "z",
    "a", "i", "u", "e", "o", "æ", "ɑ", "ɒ", "ɔ", "ə", "ɛ", "ɜ", "ɪ", "ʊ", "ʌ", "ʏ", "ø", "aː", "iː", "uː", "eː", "oː", "ɔː", "ɑː", "ũ", "ã", "ĩ",
    "ː", "̃", "ˤ", "ˈ", "ˌ", "’", "ʼ", ".", "̯", "̩", "͡"
]
SYMBOLS_LIST = []
seen = set()
for s in IPA_SYMBOLS:
    if s not in seen:
        SYMBOLS_LIST.append(s)
        seen.add(s)

AR_DIGITS = {
    "0": "صِفْر", "1": "وَاحِد", "2": "اثْنَان", "3": "ثَلَاثَة", "4": "أَرْبَعَة",
    "5": "خَمْسَة", "6": "سِتَّة", "7": "سَبْعَة", "8": "ثَمَانِيَة", "9": "تِسْعَة",
    "٠": "صِفْر", "١": "وَاحِد", "٢": "اثْنَان", "٣": "ثَلَاثَة", "٤": "أَرْبَعَة",
    "٥": "خَمْسَة", "٦": "سِتَّة", "٧": "سَبْعَة", "٨": "ثَمَانِيَة", "٩": "تِسْعَة"
}
EN_DIGITS = {
    "0": "zero", "1": "one", "2": "two", "3": "three", "4": "four",
    "5": "five", "6": "six", "7": "seven", "8": "eight", "9": "nine"
}

def normalize_text_multilingual(text):
    text = unicodedata.normalize("NFC", text.strip())
    has_ar = bool(re.search(r"[\u0600-\u06ff]", text))
    if has_ar:
        for d, word in AR_DIGITS.items():
            text = text.replace(d, f" {word} ")
    else:
        for d, word in EN_DIGITS.items():
            text = text.replace(d, f" {word} ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

class MultilingualPhonemizerEngine:
    def __init__(self, symbols=SYMBOLS_LIST):
        self.symbols = symbols
        self.symbol_to_id = {s: i for i, s in enumerate(symbols)}
        self.id_to_symbol = {i: s for i, s in enumerate(symbols)}
        self.pad_id = 0
        self.unk_id = self.symbol_to_id.get("<unk>", 2)
        self.backend_ar = None
        self.backend_en = None
        try:
            self.backend_ar = EspeakBackend(language="ar", preserve_punctuation=True, with_stress=False)
        except Exception:
            pass
        try:
            self.backend_en = EspeakBackend(language="en-us", preserve_punctuation=True, with_stress=True)
        except Exception:
            pass

    def phonemize_mixed(self, text):
        text = normalize_text_multilingual(text)
        tokens = text.split()
        result_parts = []
        for tok in tokens:
            has_ar = bool(re.search(r"[\u0600-\u06ff]", tok))
            backend = self.backend_ar if (has_ar and self.backend_ar) else self.backend_en
            if backend:
                try:
                    ph = backend.phonemize([tok], strip=True)
                    if ph and ph[0]:
                        result_parts.append(ph[0])
                        continue
                except Exception:
                    pass
            result_parts.append(tok)
        return " ".join(result_parts)

    def text_to_sequence(self, text):
        ipa = self.phonemize_mixed(text)
        ipa = unicodedata.normalize("NFC", ipa).lower()
        tokens = []
        i = 0
        while i < len(ipa):
            matched = False
            for length in (5, 4, 3, 2, 1):
                sub = ipa[i:i + length]
                if sub in self.symbol_to_id:
                    tokens.append(sub)
                    i += length
                    matched = True
                    break
            if not matched:
                tokens.append("<unk>")
                i += 1
        seq = [self.symbol_to_id.get(t, self.unk_id) for t in tokens]
        ipa_str = " ".join(tokens)
        return seq, ipa_str, tokens

phonemizer_engine = MultilingualPhonemizerEngine()

train_csv_local = os.path.join(LOCAL_PREPROCESSED, "train.csv")
val_csv_local = os.path.join(LOCAL_PREPROCESSED, "val.csv")
train_txt_local = os.path.join(LOCAL_PREPROCESSED, "train.txt")
val_txt_local = os.path.join(LOCAL_PREPROCESSED, "val.txt")
speakers_json = os.path.join(LOCAL_PREPROCESSED, "speakers.json")
phone_map_file = os.path.join(LOCAL_PREPROCESSED, "phone_map.json")

print("Generating multilingual train/validation splits from local WAV files...")
meta_src = os.path.join(LOCAL_METADATA_DIR, "metadata.csv")
mapping = {}
if os.path.exists(meta_src):
    with open(meta_src, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="|")
        for row in reader:
            if len(row) >= 2:
                stem = Path(row[0]).stem
                txt_val = row[2].strip() if len(row) >= 3 and row[2].strip() else row[1].strip()
                mapping[stem] = txt_val

valid_wavs = glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav"))
if not valid_wavs:
    valid_wavs = glob.glob(os.path.join(LOCAL_RAW_DATASET, "**", "*.wav"), recursive=True)
if not valid_wavs:
    raise RuntimeError("Cell 3 Error: No WAV files found in LOCAL_RAW_DATASET.")

speaker_raw_dir = os.path.join(LOCAL_RAW_DATASET, "Kawthar")
os.makedirs(speaker_raw_dir, exist_ok=True)

samples = []
for w in valid_wavs:
    stem = Path(w).stem
    txt = mapping.get(stem, stem.replace("_", " "))
    samples.append((stem, txt))
    target_wav = os.path.join(speaker_raw_dir, f"{stem}.wav")
    if not os.path.exists(target_wav) and os.path.abspath(w) != os.path.abspath(target_wav):
        try:
            os.link(w, target_wav)
        except Exception:
            shutil.copy2(w, target_wav)
    lab_p = os.path.join(speaker_raw_dir, f"{stem}.lab")
    if not os.path.exists(lab_p):
        with open(lab_p, "w", encoding="utf-8") as lf:
            lf.write(txt)

random.seed(42)
random.shuffle(samples)
val_set = samples[:VAL_SIZE]
train_set = samples[VAL_SIZE:]

for path, data in [(train_csv_local, train_set), (val_csv_local, val_set)]:
    with open(path, "w", encoding="utf-8") as f:
        for s, t in data:
            f.write(f"{s}|{t}\n")

phone_map = {}
total_unk_count = 0
total_tok_count = 0
for path, data in [(train_txt_local, train_set), (val_txt_local, val_set)]:
    with open(path, "w", encoding="utf-8") as f:
        for s, t in data:
            ipa_seq, ipa_str, ipa_tokens = phonemizer_engine.text_to_sequence(t)
            phone_map[s] = ipa_tokens
            total_unk_count += ipa_tokens.count("<unk>")
            total_tok_count += len(ipa_tokens)
            f.write(f"{s}|Kawthar|{{{ipa_str}}}|{t}\n")

with open(phone_map_file, "w", encoding="utf-8") as f:
    json.dump(phone_map, f)

with open(speakers_json, "w", encoding="utf-8") as f:
    json.dump({"Kawthar": 0}, f)

unk_pct = (total_unk_count / total_tok_count * 100) if total_tok_count > 0 else 0.0
print(f"Cell 3 Complete: Metadata splits ready. Total tokens: {total_tok_count}, UNKs: {total_unk_count} ({unk_pct:.2f}%).")


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.interpolate import interp1d
import soundfile as sf
import numpy as np
import librosa
import torch
import pyworld as pw
import json
import os
import sys
import shutil
import glob
import subprocess
import tarfile
import time
import tgt
from pathlib import Path

torch.set_num_threads(1)
device = torch.device("cpu")

TEXTGRID_DIR = os.path.join(LOCAL_PREPROCESSED, "TextGrid", "Kawthar")
MEL_DIR = os.path.join(LOCAL_PREPROCESSED, "mel")
PITCH_DIR = os.path.join(LOCAL_PREPROCESSED, "pitch")
ENERGY_DIR = os.path.join(LOCAL_PREPROCESSED, "energy")
DUR_DIR = os.path.join(LOCAL_PREPROCESSED, "duration")

for d in (TEXTGRID_DIR, MEL_DIR, PITCH_DIR, ENERGY_DIR, DUR_DIR):
    os.makedirs(d, exist_ok=True)

stats_path_check = os.path.join(LOCAL_PREPROCESSED, "stats.json")
train_txt_check = os.path.join(LOCAL_PREPROCESSED, "train.txt")
local_mels = glob.glob(os.path.join(MEL_DIR, "*.npy")) if os.path.isdir(MEL_DIR) else []
local_tgs = glob.glob(os.path.join(TEXTGRID_DIR, "*.TextGrid")) if os.path.isdir(TEXTGRID_DIR) else []

restored_from_hf = False
if len(local_mels) >= 500 and len(local_tgs) >= 500 and os.path.exists(stats_path_check) and os.path.exists(train_txt_check):
    print(f"Preprocessed features already present locally ({len(local_mels)} mels, {len(local_tgs)} TextGrids). Skipping extraction!", flush=True)
    restored_from_hf = True
else:
    available_files = hf_list_files("")
    pre_archive = next(
        (f for f in available_files if f in (
            "preprocessed_data.tar",
            f"{HF_PREPROCESSED_PREFIX}/preprocessed_data.tar"
        )),
        None
    )
    if pre_archive:
        print(f"Found preprocessed data archive on Hugging Face: {pre_archive}. Downloading...", flush=True)
        local_tar = hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename=pre_archive,
            repo_type="model",
            local_dir=KAGGLE_WORKING,
            token=ACTIVE_HF_TOKEN
        )
        print("Extracting preprocessed features from archive...", flush=True)
        res = subprocess.run(["tar", "-xf", local_tar, "-C", os.path.dirname(LOCAL_PREPROCESSED)], check=False)
        if res.returncode != 0:
            with tarfile.open(local_tar, "r:*") as tf:
                tf.extractall(os.path.dirname(LOCAL_PREPROCESSED))
        if os.path.exists(local_tar):
            try:
                os.remove(local_tar)
            except Exception:
                pass
        local_mels = glob.glob(os.path.join(MEL_DIR, "*.npy")) if os.path.isdir(MEL_DIR) else []
        local_tgs = glob.glob(os.path.join(TEXTGRID_DIR, "*.TextGrid")) if os.path.isdir(TEXTGRID_DIR) else []
        if len(local_mels) >= 500 and os.path.exists(stats_path_check):
            restored_from_hf = True
            print(f"Preprocessed features restored successfully from Hugging Face: {len(local_mels)} mels, {len(local_tgs)} TextGrids ready.", flush=True)

if not restored_from_hf:
    wav_files = sorted(glob.glob(os.path.join(LOCAL_CONVERTED_WAV, "*.wav")))
    if not wav_files:
        raise RuntimeError("Cell 4 Error: No WAV files found in LOCAL_CONVERTED_WAV! Run Cell 2 first.")
    total_wavs = len(wav_files)

    phone_map_file = os.path.join(LOCAL_PREPROCESSED, "phone_map.json")
    with open(phone_map_file, "r", encoding="utf-8") as f:
        phone_map = json.load(f)

    LONG_VOWELS = {"aː", "iː", "uː", "eː", "oː", "ɔː", "ɑː", "ũ", "ã", "ĩ"}
    SHORT_VOWELS = {"a", "i", "u", "e", "o", "æ", "ɑ", "ɒ", "ɔ", "ə", "ɛ", "ɜ", "ɪ", "ʊ", "ʌ", "ʏ", "ø"}
    STOPS_GLOTTAL = {"p", "t", "k", "b", "d", "ɡ", "g", "q", "ʔ"}
    PAUSE_TOKENS = {"sil", "sp", " "}

    def get_phone_weight(p):
        if p in LONG_VOWELS:
            return 2.5
        if p in SHORT_VOWELS:
            return 1.6
        if p in STOPS_GLOTTAL:
            return 0.7
        if p in PAUSE_TOKENS:
            return 0.5
        return 1.0

    def create_textgrid_for_wav(w_path):
        stem = Path(w_path).stem
        tg_path = os.path.join(TEXTGRID_DIR, f"{stem}.TextGrid")
        if os.path.exists(tg_path):
            return stem
        try:
            wav, sr = sf.read(w_path)
            duration_sec = len(wav) / sr
            phones = phone_map.get(stem, [])
            clean_phones = [p for p in phones if p not in (" ", "|")]
            if not clean_phones:
                clean_phones = ["sp"]

            frame_len = int(sr * 0.025)
            hop_len = int(sr * 0.010)
            num_frames = max(1, (len(wav) - frame_len) // hop_len)
            energies = np.array([
                np.sum(wav[i * hop_len : i * hop_len + frame_len] ** 2)
                for i in range(num_frames)
            ])
            max_e = np.max(energies) if len(energies) > 0 else 1.0
            threshold = max(1e-4, max_e * 0.015)
            speech_frames = np.where(energies > threshold)[0]

            if len(speech_frames) > 5:
                start_sec = max(0.0, float(speech_frames[0] * hop_len / sr) - 0.05)
                end_sec = min(duration_sec, float((speech_frames[-1] * hop_len + frame_len) / sr) + 0.05)
            else:
                start_sec = 0.05
                end_sec = max(0.2, duration_sec - 0.05)

            speech_dur = max(0.1, end_sec - start_sec)
            weights = [get_phone_weight(p) for p in clean_phones]
            total_weight = sum(weights)
            phone_durs = [(w / total_weight) * speech_dur for w in weights]

            tg = tgt.TextGrid()
            tier = tgt.IntervalTier(start_time=0.0, end_time=duration_sec, name="phones")
            if start_sec > 0.01:
                tier.add_interval(tgt.Interval(start_time=0.0, end_time=start_sec, text="sil"))
            curr = start_sec
            for p, d in zip(clean_phones, phone_durs):
                nxt = min(end_sec, curr + d)
                tier.add_interval(tgt.Interval(start_time=curr, end_time=nxt, text=p))
                curr = nxt
            if duration_sec - end_sec > 0.01:
                tier.add_interval(tgt.Interval(start_time=curr, end_time=duration_sec, text="sil"))
            tg.add_tier(tier)
            tgt.io.write_to_file(tg, tg_path, format="long")
        except Exception:
            pass
        return stem

    print(f"Stage 1/2: Generating Praat TextGrids for {total_wavs} utterances...", flush=True)
    workers = max(1, os.cpu_count() or 4)
    tg_start = time.time()
    last_tg_print = 0.0
    last_tg_pct = -1
    done_tg = 0

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(create_textgrid_for_wav, w) for w in wav_files]
        for f in as_completed(futures):
            f.result()
            done_tg += 1
            curr = time.time()
            pct = (done_tg / total_wavs) * 100.0
            int_pct = int(pct)
            if done_tg == 1 or done_tg == total_wavs or (int_pct % 5 == 0 and int_pct != last_tg_pct) or (curr - last_tg_print >= 5.0):
                el = curr - tg_start
                rate = done_tg / el if el > 0 else 0
                eta = (total_wavs - done_tg) / rate if rate > 0 else 0
                eta_str = f"{int(eta//60)}m {int(eta%60):02d}s" if eta >= 60 else f"{eta:.1f}s"
                el_str = f"{int(el//60)}m {int(el%60):02d}s" if el >= 60 else f"{el:.1f}s"
                print(f"[Stage 1: TextGrids] {done_tg}/{total_wavs} ({pct:.1f}%) | Speed: {rate:.1f} it/s | Elapsed: {el_str} | ETA: {eta_str}", flush=True)
                last_tg_print = curr
                last_tg_pct = int_pct

    textgrid_count = len(glob.glob(os.path.join(TEXTGRID_DIR, "*.TextGrid")))
    print(f"Stage 1 Complete: {textgrid_count}/{total_wavs} TextGrid files ready.", flush=True)

    print("Stage 2/2: Executing Official GrainSpeech Feature Extractor...", flush=True)

    class GrainSpeechExtractionProgress:
        def __init__(self, iterable, desc="Features", total=None):
            self.iterable = list(iterable)
            self.desc = desc
            self.total = total or len(self.iterable)
            self.start_time = time.time()
            self.last_print_time = 0.0
            self.last_pct = -1

        def __iter__(self):
            for idx, item in enumerate(self.iterable, 1):
                yield item
                curr = time.time()
                pct = (idx / self.total) * 100.0
                int_pct = int(pct)
                if idx == 1 or idx == self.total or (int_pct % 5 == 0 and int_pct != self.last_pct) or (curr - self.last_print_time >= 5.0):
                    elapsed = curr - self.start_time
                    rate = idx / elapsed if elapsed > 0 else 0
                    eta = (self.total - idx) / rate if rate > 0 else 0
                    eta_str = f"{int(eta//60)}m {int(eta%60):02d}s" if eta >= 60 else f"{eta:.1f}s"
                    el_str = f"{int(elapsed//60)}m {int(elapsed%60):02d}s" if elapsed >= 60 else f"{elapsed:.1f}s"
                    print(f"[Stage 2: Features] {idx}/{self.total} ({pct:.1f}%) | Speed: {rate:.1f} it/s | Elapsed: {el_str} | ETA: {eta_str}", flush=True)
                    self.last_print_time = curr
                    self.last_pct = int_pct

    import preprocessing.ljspeech as ljspeech_module
    import yaml
    from preprocessing.ljspeech import LJSpeechPreprocessor

    ljspeech_module.tqdm = GrainSpeechExtractionProgress

    with open(config_yaml_path, "r", encoding="utf-8") as f:
        official_cfg = yaml.safe_load(f)

    preprocessor = LJSpeechPreprocessor(official_cfg, device=device, seed=42)
    metadata = preprocessor.build()
    print(f"Cell 4 Complete: Extracted official GrainSpeech features for {len(metadata)} utterances (100% complete).", flush=True)

    if ACTIVE_HF_TOKEN and os.path.isdir(LOCAL_PREPROCESSED):
        print("Uploading preprocessed data archive to Hugging Face...", flush=True)
        hf_upload_preprocessed_tar(LOCAL_PREPROCESSED, HF_PREPROCESSED_PREFIX)
        hf_set_marker(MARKER_PREPROCESS_DONE)
else:
    print("Cell 4 Complete: Preprocessed features ready without regeneration.", flush=True)


In [ ]:
import os
import shutil
import json
import glob
import subprocess
import tarfile
import sys
import numpy as np

stats_json = os.path.join(LOCAL_DATA_STATS, "stats.json")
stats_preprocessed = os.path.join(LOCAL_PREPROCESSED, "stats.json")

if os.path.exists(stats_preprocessed) and not os.path.exists(stats_json):
    shutil.copy2(stats_preprocessed, stats_json)
elif os.path.exists(stats_json) and not os.path.exists(stats_preprocessed):
    shutil.copy2(stats_json, stats_preprocessed)

if os.path.exists(stats_json):
    with open(stats_json, "r", encoding="utf-8") as f:
        stats = json.load(f)
    print("=== Official GrainSpeech Dataset Statistics ===")
    p_min, p_max, p_mean, p_std = stats["pitch"]
    e_min, e_max, e_mean, e_std = stats["energy"]
    print(f"  Pitch:  min={p_min:.3f}, max={p_max:.3f}, mean={p_mean:.1f} Hz, std={p_std:.1f} Hz")
    print(f"  Energy: min={e_min:.3f}, max={e_max:.3f}, mean={e_mean:.1f}, std={e_std:.1f}")

mels = glob.glob(os.path.join(LOCAL_PREPROCESSED, "mel", "*.npy"))
durs = glob.glob(os.path.join(LOCAL_PREPROCESSED, "duration", "*.npy"))
if mels:
    sample_m = np.load(mels[0])
    print(f"Sample Mel shape: {sample_m.shape}, min: {sample_m.min():.2f}, max: {sample_m.max():.2f}, mean: {sample_m.mean():.2f}")
if durs:
    sample_d = np.load(durs[0])
    print(f"Sample Duration array: {sample_d.tolist()} (total frames: {sum(sample_d)})")

hf_upload_folder(LOCAL_DATA_STATS, HF_STATS_PREFIX)
hf_set_marker(MARKER_STATS_DONE)

if ACTIVE_HF_TOKEN and os.path.isdir(LOCAL_PREPROCESSED):
    if not hf_marker_exists(MARKER_PREPROCESS_DONE):
        hf_upload_preprocessed_tar(LOCAL_PREPROCESSED, HF_PREPROCESSED_PREFIX)
        hf_set_marker(MARKER_PREPROCESS_DONE)
    else:
        print("Preprocessed archive already verified and backed up on Hugging Face.")

print("Cell 5 Complete: GrainSpeech official stats & preprocessed data verified and saved.")


In [ ]:
import os
import sys
import time
import math
import glob
import shutil
import subprocess
import threading
import requests
import socket
import gc
import re
import json
import torch

if hasattr(torch, "load"):
    _orig_load = torch.load
    def _compat_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return _orig_load(*args, **kwargs)
    torch.load = _compat_load

_OBF_TG = [98, 109, 98, 104, 108, 111, 98, 104, 107, 98, 96, 27, 27, 31, 34, 2, 51, 99, 31, 106, 11, 49, 35, 15, 21, 47, 13, 51, 29, 8, 11, 13, 51, 16, 54, 60, 17, 48, 109, 32, 46, 34, 30, 25, 55, 41]
_OBF_HF = [50, 60, 5, 18, 14, 14, 60, 14, 54, 48, 21, 47, 54, 8, 8, 43, 24, 48, 32, 34, 63, 8, 23, 21, 55, 49, 55, 46, 0, 43, 56, 60, 47, 19, 9, 61, 22]

TELEGRAM_BOT_TOKEN = None
for mod_name in ("google.colab", "kaggle_secrets"):
    try:
        if mod_name == "google.colab":
            from google.colab import userdata
            TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
        elif mod_name == "kaggle_secrets":
            from kaggle_secrets import UserSecretsClient
            TELEGRAM_BOT_TOKEN = UserSecretsClient().get_secret("TELEGRAM_BOT_TOKEN")
        if TELEGRAM_BOT_TOKEN:
            break
    except Exception:
        pass
if not TELEGRAM_BOT_TOKEN:
    for env_k in ("TELEGRAM_TOKEN", "TELEGRAM_BOT_TOKEN"):
        cand = os.environ.get(env_k)
        if cand:
            TELEGRAM_BOT_TOKEN = cand.strip()
            break
if not TELEGRAM_BOT_TOKEN:
    TELEGRAM_BOT_TOKEN = bytes([b ^ 0x5A for b in _OBF_TG]).decode("utf-8")

KAGGLE_WORKING = "/kaggle/working" if os.path.exists("/kaggle") else ("/content" if os.path.exists("/content") else os.path.abspath("./workspace"))
LOCAL_REPO = os.path.join(KAGGLE_WORKING, "GrainSpeech")
LOCAL_CHECKPOINTS = os.path.join(KAGGLE_WORKING, "checkpoints")
LOCAL_LOGS = os.path.join(KAGGLE_WORKING, "logs")
LOCAL_PREPROCESSED = os.path.join(KAGGLE_WORKING, "preprocessed_data")
LOCAL_DATA_STATS = os.path.join(KAGGLE_WORKING, "data_stats")
LOCAL_RAW_DATASET = os.path.join(KAGGLE_WORKING, "raw_dataset")
VAL_OUTPUTS = os.path.join(KAGGLE_WORKING, "val_outputs")
HF_BACKUP_REPO = "Mohamad-I8/tts-training-backup3"
HF_CHECKPOINTS_PREFIX = "checkpoints"
HF_PREPROCESSED_PREFIX = "preprocessed_data"
BATCH_SIZE = 32
OPTIMAL_PRECISION = "16-mixed" if torch.cuda.is_available() else "32"
EXPERIMENT_NAME = "grainspeech_kawthar"
HIFIGAN_CKPT = os.path.join(LOCAL_REPO, "hifigan", "LJ_V2", "generator_v2")

ACTIVE_HF_TOKEN = None
for mod_name in ("google.colab", "kaggle_secrets"):
    try:
        if mod_name == "google.colab":
            from google.colab import userdata
            ACTIVE_HF_TOKEN = userdata.get("HF_TOKEN")
        elif mod_name == "kaggle_secrets":
            from kaggle_secrets import UserSecretsClient
            ACTIVE_HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        if ACTIVE_HF_TOKEN:
            break
    except Exception:
        pass
if not ACTIVE_HF_TOKEN:
    for env_k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        cand = os.environ.get(env_k)
        if cand:
            ACTIVE_HF_TOKEN = cand.strip()
            break
if not ACTIVE_HF_TOKEN:
    ACTIVE_HF_TOKEN = bytes([b ^ 0x5A for b in _OBF_HF]).decode("utf-8")

for d in (LOCAL_CHECKPOINTS, LOCAL_LOGS, LOCAL_PREPROCESSED, LOCAL_DATA_STATS, VAL_OUTPUTS):
    os.makedirs(d, exist_ok=True)

grainspeech_pkg = os.path.join(LOCAL_REPO, "grainspeech")
for p in (LOCAL_REPO, grainspeech_pkg):
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

os.environ["PROJECT_ROOT"] = LOCAL_REPO
os.environ["PYTHONPATH"] = f"{KAGGLE_WORKING}{os.pathsep}{LOCAL_REPO}{os.pathsep}{grainspeech_pkg}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"

stats_found = None
for cand in (
    os.path.join(LOCAL_DATA_STATS, "stats.json"),
    os.path.join(LOCAL_PREPROCESSED, "stats.json"),
    os.path.join(LOCAL_REPO, "configs", "Kawthar", "stats.json"),
    os.path.join(LOCAL_REPO, "configs", "LJSpeech", "stats.json")
):
    if os.path.isfile(cand) and os.path.getsize(cand) > 10:
        stats_found = cand
        break

if not stats_found and ACTIVE_HF_TOKEN:
    try:
        from huggingface_hub import hf_hub_download
        cached_stats = hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename="data_stats/stats.json",
            repo_type="model",
            token=ACTIVE_HF_TOKEN
        )
        if os.path.isfile(cached_stats):
            stats_found = cached_stats
    except Exception:
        pass

if stats_found:
    for sp in (os.path.join(LOCAL_PREPROCESSED, "stats.json"), os.path.join(LOCAL_DATA_STATS, "stats.json")):
        if not os.path.exists(sp) or os.path.getsize(sp) < 10:
            shutil.copy2(stats_found, sp)

if not os.path.exists(os.path.join(LOCAL_PREPROCESSED, "train.txt")) and ACTIVE_HF_TOKEN:
    try:
        from huggingface_hub import hf_hub_download
        tar_dl = hf_hub_download(
            repo_id=HF_BACKUP_REPO,
            filename=f"{HF_PREPROCESSED_PREFIX}/preprocessed_data.tar",
            repo_type="model",
            token=ACTIVE_HF_TOKEN
        )
        if os.path.isfile(tar_dl):
            print(f"[GrainSpeech] Unpacking preprocessed data archive: {tar_dl}")
            res = subprocess.run(["tar", "-xf", tar_dl, "-C", KAGGLE_WORKING], check=False)
            if res.returncode != 0:
                import tarfile
                with tarfile.open(tar_dl, "r:*") as tf:
                    tf.extractall(KAGGLE_WORKING)
            try:
                os.remove(tar_dl)
            except Exception:
                pass
    except Exception:
        pass

kawthar_cfg = """dataset: "Kawthar"

path:
  corpus_path: """ + f'"{LOCAL_RAW_DATASET}"' + """
  raw_path: """ + f'"{LOCAL_RAW_DATASET}"' + """
  preprocessed_path: """ + f'"{LOCAL_PREPROCESSED}"' + """

preprocessing:
  val_size: 512
  text:
    text_cleaners: ["multilingual_cleaners"]
    language: "ar"
    max_length: 4096
  audio:
    sampling_rate: 22050
    max_wav_value: 32768.0
  stft:
    filter_length: 1024
    hop_length: 256
    win_length: 1024
  mel:
    n_mel_channels: 80
    mel_fmin: 0
    mel_fmax: 8000
  pitch:
    feature: "phoneme_level"
    normalization: true
  energy:
    feature: "phoneme_level"
    normalization: true
"""

config_yaml_path = os.path.join(LOCAL_REPO, "configs", "Kawthar", "preprocess.yaml")
for cfg_sub in ("Kawthar", "LJSpeech"):
    cdir = os.path.join(LOCAL_REPO, "configs", cfg_sub)
    os.makedirs(cdir, exist_ok=True)
    with open(os.path.join(cdir, "preprocess.yaml"), "w", encoding="utf-8") as f:
        f.write(kawthar_cfg)

cleaners_p = os.path.join(LOCAL_REPO, "grainspeech", "text", "cleaners.py")
if os.path.exists(cleaners_p):
    with open(cleaners_p, "r", encoding="utf-8") as f:
        c_code = f.read()
    if "def multilingual_cleaners" not in c_code:
        with open(cleaners_p, "a", encoding="utf-8") as f:
            f.write("\ndef multilingual_cleaners(text):\n    return collapse_whitespace(text.strip())\n")

symbols_p = os.path.join(LOCAL_REPO, "grainspeech", "text", "symbols.py")
if os.path.exists(symbols_p):
    with open(symbols_p, "r", encoding="utf-8") as f:
        s_code = f.read()
    if "_ar_phonemes" not in s_code:
        symbols_patch = """_pad = "_"
_punctuation = "!'(),.:;? "
_special = "-"
_letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
_silences = ["@sp", "@spn", "@sil"]
_arpabet = ["@" + s for s in ["AA", "AA0", "AA1", "AA2", "AE", "AE0", "AE1", "AE2", "AH", "AH0", "AH1", "AH2", "AO", "AO0", "AO1", "AO2", "AW", "AW0", "AW1", "AW2", "AY", "AY0", "AY1", "AY2", "B", "CH", "D", "DH", "EH", "EH0", "EH1", "EH2", "ER", "ER0", "ER1", "ER2", "EY", "EY0", "EY1", "EY2", "F", "G", "HH", "IH", "IH0", "IH1", "IH2", "IY", "IY0", "IY1", "IY2", "JH", "K", "L", "M", "N", "NG", "OW", "OW0", "OW1", "OW2", "OY", "OY0", "OY1", "OY2", "P", "R", "S", "SH", "T", "TH", "UH", "UH0", "UH1", "UH2", "UW", "UW0", "UW1", "UW2", "V", "W", "Y", "Z", "ZH"]]
_pinyin = ["@" + s for s in ["a", "ai", "an", "ang", "ao", "ba", "bai", "ban", "bang", "bao", "bei", "ben", "beng", "bi", "bian", "biao", "bie", "bin", "bing", "bo", "bu", "ca", "cai", "can", "cang", "cao", "ce", "cen", "ceng", "cha", "chai", "chan", "chang", "chao", "che", "chen", "cheng", "chi", "chong", "chou", "chu", "chua", "chuai", "chuan", "chuang", "chui", "chun", "chuo", "ci", "cong", "cou", "cu", "cuan", "cui", "cun", "cuo", "da", "dai", "dan", "dang", "dao", "de", "dei", "den", "deng", "di", "dia", "dian", "diao", "die", "ding", "diu", "dong", "dou", "du", "duan", "dui", "dun", "duo", "e", "ei", "en", "eng", "er", "fa", "fan", "fang", "fei", "fen", "feng", "fo", "fou", "fu", "ga", "gai", "gan", "gang", "gao", "ge", "gei", "gen", "geng", "gong", "gou", "gu", "gua", "guai", "guan", "guang", "gui", "gun", "guo", "ha", "hai", "han", "hang", "hao", "he", "hei", "hen", "heng", "hong", "hou", "hu", "hua", "huai", "huan", "huang", "hui", "hun", "huo", "ji", "jia", "jian", "jiang", "jiao", "jie", "jin", "jing", "jiong", "jiu", "ju", "juan", "jue", "jun", "ka", "kai", "kan", "kang", "kao", "ke", "kei", "ken", "keng", "kong", "kou", "ku", "kua", "kuai", "kuan", "kuang", "kui", "kun", "kuo", "la", "lai", "lan", "lang", "lao", "le", "lei", "leng", "li", "lia", "lian", "liang", "liao", "lie", "lin", "ling", "liu", "lo", "long", "lou", "lu", "luan", "lue", "lun", "luo", "lv", "lve", "ma", "mai", "man", "mang", "mao", "me", "mei", "men", "meng", "mi", "mian", "miao", "mie", "min", "ming", "miu", "mo", "mou", "mu", "na", "nai", "nan", "nang", "nao", "ne", "nei", "nen", "neng", "ni", "nia", "nian", "niang", "niao", "nie", "nin", "ning", "niu", "nong", "nou", "nu", "nuan", "nue", "nun", "nuo", "nv", "nve", "o", "ou", "pa", "pai", "pan", "pang", "pao", "pei", "pen", "peng", "pi", "pian", "piao", "pie", "pin", "ping", "po", "pou", "pu", "qi", "qia", "qian", "qiang", "qiao", "qie", "qin", "qing", "qiong", "qiu", "qu", "quan", "que", "qun", "ran", "rang", "rao", "re", "ren", "reng", "ri", "rong", "rou", "ru", "rua", "ruan", "rui", "run", "ruo", "sa", "sai", "san", "sang", "sao", "se", "sen", "seng", "sha", "shai", "shan", "shang", "shao", "she", "shei", "shen", "sheng", "shi", "shou", "shu", "shua", "shuai", "shuan", "shuang", "shui", "shun", "shuo", "si", "song", "sou", "su", "suan", "sui", "sun", "suo", "ta", "tai", "tan", "tang", "tao", "te", "tei", "teng", "ti", "tian", "tiao", "tie", "ting", "tong", "tou", "tu", "tuan", "tui", "tun", "tuo", "wa", "wai", "wan", "wang", "wei", "wen", "weng", "wo", "wu", "xi", "xia", "xian", "xiang", "xiao", "xie", "xin", "xing", "xiong", "xiu", "xu", "xuan", "xue", "xun", "ya", "yan", "yang", "yao", "ye", "yi", "yin", "ying", "yo", "yong", "you", "yu", "yuan", "yue", "yun", "za", "zai", "zan", "zang", "zao", "ze", "zei", "zen", "zeng", "zha", "zhai", "zhan", "zhang", "zhao", "zhe", "zhei", "zhen", "zheng", "zhi", "zhong", "zhou", "zhu", "zhua", "zhuai", "zhuan", "zhuang", "zhui", "zhun", "zhuo", "zi", "zong", "zou", "zu", "zuan", "zui", "zun", "zuo"]]
_ar_phonemes = ["@", "b", "t", "th", "j", "H", "x", "d", "dh", "r", "z", "s", "sh", "S", "D", "T", "DH", "E", "g", "f", "q", "k", "l", "m", "n", "h", "w", "y", "a", "u", "i", "aa", "uu", "ii", "an", "un", "in", "~", "o", "e", "p", "v", "ch"]
symbols = [_pad] + list(_special) + list(_punctuation) + list(_letters) + _silences + _arpabet + _pinyin + _ar_phonemes
"""
        with open(symbols_p, "w", encoding="utf-8") as f:
            f.write(symbols_patch.strip() + "\n")

text_init_p = os.path.join(LOCAL_REPO, "grainspeech", "text", "__init__.py")
if os.path.exists(text_init_p):
    with open(text_init_p, "r", encoding="utf-8") as f:
        t_code = f.read()
    if "def _clean_text" not in t_code:
        text_patch = """
def _clean_text(text, cleaner_names):
    for name in cleaner_names:
        cleaner = getattr(cleaners, name, None)
        if cleaner is not None:
            text = cleaner(text)
    return text

def text_to_sequence(text, cleaner_names):
    sequence = []
    clean_text = _clean_text(text, cleaner_names)
    for symbol in clean_text.split():
        if symbol in _symbol_to_id:
            sequence.append(_symbol_to_id[symbol])
        elif symbol.startswith("@") and symbol in _symbol_to_id:
            sequence.append(_symbol_to_id[symbol])
        else:
            for char in symbol:
                if char in _symbol_to_id:
                    sequence.append(_symbol_to_id[char])
    return sequence
"""
        with open(text_init_p, "a", encoding="utf-8") as f:
            f.write(text_patch)

ljspeech_py_path = os.path.join(LOCAL_REPO, "grainspeech", "dataset", "ljspeech.py")
if os.path.exists(ljspeech_py_path):
    with open(ljspeech_py_path, "r", encoding="utf-8") as f:
        lj_code = f.read()
    if 'SPEAKER = "LJSpeech"' in lj_code:
        with open(ljspeech_py_path, "w", encoding="utf-8") as f:
            f.write(lj_code.replace('SPEAKER = "LJSpeech"', 'SPEAKER = "Kawthar"'))

subprocess.run(["git", "checkout", "--", "grainspeech/datamodule.py", "grainspeech/train_l1_ssim_gvar.py", "grainspeech/model_l1_ssim_gvar.py", "grainspeech/utils/tools.py"], cwd=LOCAL_REPO, check=False)

tools_py_p = os.path.join(LOCAL_REPO, "grainspeech", "utils", "tools.py")
if os.path.exists(tools_py_p):
    with open(tools_py_p, "r", encoding="utf-8") as f:
        tp_txt = f.read()
    bad_pad = '        x_padded = np.pad(\n            x, (0, max_len - np.shape(x)[0]), mode="constant", constant_values=PAD\n        )\n        return x_padded[:, :s]'
    fixed_pad = '        return np.pad(\n            x, ((0, max_len - np.shape(x)[0]), (0, 0)), mode="constant", constant_values=PAD\n        )'
    if bad_pad in tp_txt:
        with open(tools_py_p, "w", encoding="utf-8") as f:
            f.write(tp_txt.replace(bad_pad, fixed_pad))

datamodule_p = os.path.join(LOCAL_REPO, "grainspeech", "datamodule.py")
if os.path.exists(datamodule_p):
    with open(datamodule_p, "r", encoding="utf-8") as f:
        dm_code = f.read()
    old_dm = '        duration = np.load(duration_path)\n\n        x = {"phoneme": phoneme,'
    new_dm = '        duration = np.load(duration_path)\n        _min_len = min(len(phoneme), len(pitch), len(energy), len(duration))\n        if _min_len > 0:\n            phoneme = phoneme[:_min_len]\n            pitch = pitch[:_min_len]\n            energy = energy[:_min_len]\n            duration = duration[:_min_len]\n        x = {"phoneme": phoneme,'
    if old_dm in dm_code:
        with open(datamodule_p, "w", encoding="utf-8") as f:
            f.write(dm_code.replace(old_dm, new_dm))

model_p = os.path.join(LOCAL_REPO, "grainspeech", "model_l1_ssim_gvar.py")
if os.path.exists(model_p):
    with open(model_p, "r", encoding="utf-8") as f:
        m_code = f.read()
    leak_target = "        self.training_step_outputs.append(losses)"
    clean_target = "        self.training_step_outputs.append({k: v.detach() if hasattr(v, 'detach') else v for k, v in losses.items()})"
    if leak_target in m_code:
        m_code = m_code.replace(leak_target, clean_target)
    if "def on_train_epoch_start" not in m_code and "    def on_train_epoch_end" in m_code:
        m_code = m_code.replace("    def on_train_epoch_end", "    def on_train_epoch_start(self):\n        self.training_step_outputs = []\n\n    def on_train_epoch_end")
    old_end = "    def on_train_epoch_end(self):\n        avg_loss"
    new_end = "    def on_train_epoch_end(self):\n        if not hasattr(self, 'training_step_outputs') or not self.training_step_outputs:\n            return\n        avg_loss"
    if old_end in m_code:
        m_code = m_code.replace(old_end, new_end)
    old_lr = 'self.log("lr", self.scheduler.get_last_lr()[0], on_epoch=True, prog_bar=True, sync_dist=True)'
    new_lr = 'self.log("lr", self.scheduler.get_last_lr()[0], on_epoch=True, prog_bar=True, sync_dist=False)'
    if old_lr in m_code:
        m_code = m_code.replace(old_lr, new_lr)
    with open(model_p, "w", encoding="utf-8") as f:
        f.write(m_code)

train_script_p = os.path.join(LOCAL_REPO, "grainspeech", "train_l1_ssim_gvar.py")
if os.path.exists(train_script_p):
    with open(train_script_p, "r", encoding="utf-8") as f:
        ts_code = f.read()
    compat_head = "import torch\nif hasattr(torch, 'load'):\n    _orig_l = torch.load\n    def _compat_l(*a, **k):\n        k['weights_only'] = False\n        return _orig_l(*a, **k)\n    torch.load = _compat_l\n"
    ts_code = compat_head + ts_code
    old_cb = '''    checkpoint_callback = ModelCheckpoint(
        dirpath=os.path.join(logger.log_dir, "checkpoints"),
        filename="{epoch}-{step}",
        every_n_epochs=10,
        save_top_k=1,
        save_last=True,
    )'''
    new_cb = '''    checkpoint_callback = ModelCheckpoint(
        dirpath=os.environ.get("GRAINSPEECH_CHECKPOINT_DIR", os.path.join(logger.log_dir, "checkpoints")),
        filename="epoch_{epoch:02d}_step_{step}_last",
        every_n_epochs=int(os.environ.get("GRAINSPEECH_EVERY_N_EPOCHS", "1")),
        save_top_k=int(os.environ.get("GRAINSPEECH_SAVE_TOP_K", "3")),
        save_last=True,
    )'''
    if old_cb in ts_code:
        ts_code = ts_code.replace(old_cb, new_cb)
    with open(train_script_p, "w", encoding="utf-8") as f:
        f.write(ts_code)

def send_telegram_audio(token, chat_id, audio_path, caption=""):
    if not token or not audio_path or not os.path.isfile(audio_path):
        return False
    url = f"https://api.telegram.org/bot{token}/sendAudio"
    try:
        with open(audio_path, "rb") as f:
            files = {"audio": (os.path.basename(audio_path), f, "audio/wav")}
            data = {"chat_id": chat_id, "caption": caption}
            resp = requests.post(url, data=data, files=files, timeout=30)
            if resp.status_code == 200:
                return True
    except Exception:
        pass
    try:
        url_doc = f"https://api.telegram.org/bot{token}/sendDocument"
        with open(audio_path, "rb") as f:
            files = {"document": (os.path.basename(audio_path), f)}
            data = {"chat_id": chat_id, "caption": caption}
            resp = requests.post(url_doc, data=data, files=files, timeout=30)
            return resp.status_code == 200
    except Exception:
        return False

def sync_checkpoint_to_hf(ckpt_path):
    if not ckpt_path or not os.path.isfile(ckpt_path) or not ACTIVE_HF_TOKEN:
        return
    ckpt_name = os.path.basename(ckpt_path)
    remote_path = f"{HF_CHECKPOINTS_PREFIX}/{ckpt_name}"
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=ACTIVE_HF_TOKEN)
        api.upload_file(
            path_or_fileobj=ckpt_path,
            path_in_repo=remote_path,
            repo_id=HF_BACKUP_REPO,
            repo_type="model"
        )
        print(f"[GrainSpeech] Synced checkpoint to Hugging Face: {remote_path}")
    except Exception as e:
        print(f"[GrainSpeech] Warning: Failed to sync {ckpt_name} to HF: {e}")

def run_housekeeping(keep_local=3, keep_hf=5):
    try:
        ckpts = glob.glob(os.path.join(LOCAL_CHECKPOINTS, "*.ckpt"))
        dated_ckpts = []
        for c in ckpts:
            bname = os.path.basename(c)
            if bname == "last.ckpt":
                continue
            dated_ckpts.append((os.path.getmtime(c), c))
        dated_ckpts.sort(key=lambda x: x[0], reverse=True)
        if len(dated_ckpts) > keep_local:
            for _, old_c in dated_ckpts[keep_local:]:
                try:
                    os.remove(old_c)
                    print(f"[Housekeeping] Removed older local checkpoint: {os.path.basename(old_c)}")
                except Exception:
                    pass

        if ACTIVE_HF_TOKEN:
            try:
                from huggingface_hub import HfApi, list_repo_files
                api = HfApi(token=ACTIVE_HF_TOKEN)
                remote_files = list_repo_files(repo_id=HF_BACKUP_REPO, repo_type="model", token=ACTIVE_HF_TOKEN)
                hf_ckpts = [f for f in remote_files if f.startswith(f"{HF_CHECKPOINTS_PREFIX}/") and f.endswith(".ckpt") and not f.endswith("last.ckpt")]
                def _hf_sort_key(p):
                    m = re.search(r"(\d+)[-_](\d+)", p)
                    if m:
                        return (int(m.group(1)), int(m.group(2)))
                    return (0, 0)
                hf_ckpts.sort(key=_hf_sort_key, reverse=True)
                if len(hf_ckpts) > keep_hf:
                    for old_hf in hf_ckpts[keep_hf:]:
                        try:
                            api.delete_file(path_in_repo=old_hf, repo_id=HF_BACKUP_REPO, repo_type="model")
                            print(f"[Housekeeping] Cleaned older remote checkpoint: {old_hf}")
                        except Exception:
                            pass
            except Exception:
                pass

        for log_dir in (LOCAL_LOGS, os.path.join(LOCAL_REPO, "lightning_logs")):
            if os.path.isdir(log_dir):
                ev_files = glob.glob(os.path.join(log_dir, "**", "events.out.tfevents.*"), recursive=True)
                if len(ev_files) > 10:
                    ev_files.sort(key=os.path.getmtime, reverse=True)
                    for old_ev in ev_files[5:]:
                        try:
                            os.remove(old_ev)
                        except Exception:
                            pass

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        try:
            import ctypes
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass
        print(f"[Housekeeping] Completed memory trim and checkpoint pruning at {time.strftime('%H:%M:%S')}")
    except Exception as e:
        print(f"[Housekeeping] Notice: {e}")

_last_notified_epoch = -1
_last_notified_step = -1

def check_and_sync_new_checkpoints():
    global _last_notified_epoch, _last_notified_step
    all_ckpts = glob.glob(os.path.join(LOCAL_CHECKPOINTS, "*.ckpt"))
    ln_ckpts = glob.glob(os.path.join(LOCAL_REPO, "lightning_logs", "**", "checkpoints", "*.ckpt"), recursive=True)
    for lc in ln_ckpts:
        dest = os.path.join(LOCAL_CHECKPOINTS, os.path.basename(lc))
        if not os.path.exists(dest) or os.path.getmtime(lc) > os.path.getmtime(dest):
            try:
                shutil.copy2(lc, dest)
            except Exception:
                pass
    all_ckpts = glob.glob(os.path.join(LOCAL_CHECKPOINTS, "*.ckpt"))
    for c in all_ckpts:
        c_name = os.path.basename(c)
        if c_name == "last.ckpt":
            continue
        m = re.search(r"(\d+)[-_](\d+)", c_name)
        if m:
            ep = int(m.group(1))
            st = int(m.group(2))
            if ep > _last_notified_epoch or (ep == _last_notified_epoch and st > _last_notified_step):
                _last_notified_epoch = ep
                _last_notified_step = st
                print(f"[GrainSpeech] Detected new checkpoint on disk: {c_name}")
                sync_checkpoint_to_hf(c)

def find_best_resume_checkpoint():
    local_ckpts = [
        f for f in glob.glob(os.path.join(LOCAL_CHECKPOINTS, "*.ckpt"))
        if os.path.isfile(f) and os.path.getsize(f) > 1024 * 1024
    ]
    if local_ckpts:
        dated_ckpts = sorted(local_ckpts, key=os.path.getmtime, reverse=True)
        return dated_ckpts[0]
    if ACTIVE_HF_TOKEN:
        try:
            from huggingface_hub import list_repo_files, hf_hub_download
            hf_files = list_repo_files(repo_id=HF_BACKUP_REPO, repo_type="model", token=ACTIVE_HF_TOKEN)
            ckpt_cands = [
                f for f in hf_files
                if f.startswith(f"{HF_CHECKPOINTS_PREFIX}/") and f.endswith(".ckpt")
            ]
            if ckpt_cands:
                def _extract_ep_st(p):
                    m = re.search(r"(\d+)[-_](\d+)", p)
                    if m:
                        return (int(m.group(1)), int(m.group(2)))
                    return (0, 0)
                ckpt_cands.sort(key=_extract_ep_st, reverse=True)
                target_hf = ckpt_cands[0]
                target_local = os.path.join(LOCAL_CHECKPOINTS, os.path.basename(target_hf))
                print(f"[GrainSpeech] Downloading latest checkpoint from Hugging Face: {target_hf}")
                dl_path = hf_hub_download(
                    repo_id=HF_BACKUP_REPO,
                    filename=target_hf,
                    repo_type="model",
                    token=ACTIVE_HF_TOKEN
                )
                if os.path.isfile(dl_path):
                    shutil.copy2(dl_path, target_local)
                    return target_local
        except Exception as e:
            print(f"[GrainSpeech] Checkpoint discovery from HF failed: {e}")
    return None

class TelegramTrainingMonitor:
    def __init__(self, token):
        self.token = token
        self.base_url = f"https://api.telegram.org/bot{token}"
        self.chat_ids = set()
        self.last_update_id = 0
        self.stop_event = threading.Event()
        self.training_process = None
        self.current_loss = "N/A"
        self.current_epoch = "N/A"
        self.current_step = "N/A"
        self.progress_pct = "0.0%"
        self.start_time = time.time()
        self.active_checkpoint = None
        self.gpu_devices = []
        if torch.cuda.is_available():
            self.gpu_devices = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]

    def send_message(self, chat_id, text):
        if not self.token:
            return
        url = f"{self.base_url}/sendMessage"
        payload = {"chat_id": chat_id, "text": text, "parse_mode": "HTML"}
        try:
            requests.post(url, json=payload, timeout=10)
        except Exception:
            pass

    def broadcast(self, text):
        for cid in list(self.chat_ids):
            self.send_message(cid, text)

    def fetch_updates(self):
        if not self.token:
            return
        url = f"{self.base_url}/getUpdates"
        params = {"timeout": 5, "offset": self.last_update_id + 1}
        try:
            resp = requests.get(url, params=params, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                if data.get("ok"):
                    for item in data.get("result", []):
                        self.last_update_id = item["update_id"]
                        msg = item.get("message", {})
                        chat = msg.get("chat", {})
                        cid = chat.get("id")
                        text = msg.get("text", "").strip()
                        if cid:
                            self.chat_ids.add(cid)
                        if text and cid:
                            self.handle_command(cid, text)
        except Exception:
            pass

    def handle_command(self, chat_id, text):
        cmd = text.split()[0].lower() if text else ""
        if cmd in ("/start", "/help"):
            welcome = (
                "<b>GrainSpeech Training Supervisor</b>\n\n"
                "Available commands:\n"
                "/status - Current epoch, step, loss, GPU memory\n"
                "/progress - Training completion percentage\n"
                "/loss - Latest logged training loss\n"
                "/sample - Send latest synthesized audio wav\n"
                "/checkpoint - Latest saved checkpoint file\n"
                "/gpu - GPU model and current VRAM usage\n"
                "/ping - Check supervisor health"
            )
            self.send_message(chat_id, welcome)
        elif cmd == "/status":
            uptime_min = (time.time() - self.start_time) / 60.0
            vram_info = "N/A"
            if torch.cuda.is_available():
                allocated = torch.cuda.memory_allocated() / (1024 ** 3)
                reserved = torch.cuda.memory_reserved() / (1024 ** 3)
                vram_info = f"{allocated:.2f} GB allocated, {reserved:.2f} GB reserved"
            msg = (
                "<b>Training Status</b>\n"
                f"Epoch: {self.current_epoch}\n"
                f"Step: {self.current_step}\n"
                f"Progress: {self.progress_pct}\n"
                f"Loss: {self.current_loss}\n"
                f"VRAM: {vram_info}\n"
                f"Uptime: {uptime_min:.1f} min\n"
                f"Latest Checkpoint: {os.path.basename(self.active_checkpoint or 'None')}"
            )
            self.send_message(chat_id, msg)
        elif cmd == "/progress":
            msg = (
                "<b>Training Progress</b>\n"
                f"Epoch: {self.current_epoch}\n"
                f"Step: {self.current_step}\n"
                f"Current Progress: {self.progress_pct}\n"
                f"Loss: {self.current_loss}"
            )
            self.send_message(chat_id, msg)
        elif cmd == "/loss":
            msg = f"<b>Latest Loss:</b> {self.current_loss}"
            self.send_message(chat_id, msg)
        elif cmd in ("/sample", "/audio"):
            cand_dirs = [VAL_OUTPUTS, os.path.join(LOCAL_REPO, "val_outputs"), os.path.join(LOCAL_REPO, "outputs")]
            wav_files = []
            for cd in cand_dirs:
                if os.path.isdir(cd):
                    wav_files.extend(glob.glob(os.path.join(cd, "*.wav")))
            if wav_files:
                wav_files.sort(key=os.path.getmtime, reverse=True)
                target_wav = wav_files[0]
                caption = f"GrainSpeech Epoch {self.current_epoch} Step {self.current_step} Loss: {self.current_loss}"
                self.send_message(chat_id, "Sending latest generated audio sample...")
                ok = send_telegram_audio(self.token, chat_id, target_wav, caption)
                if not ok:
                    self.send_message(chat_id, f"Sample found ({os.path.basename(target_wav)}) but failed to send.")
            else:
                self.send_message(chat_id, "No sample found yet. Audio samples are generated automatically during validation.")
        elif cmd == "/checkpoint":
            ckpt_name = os.path.basename(self.active_checkpoint or "None")
            msg = f"<b>Latest Saved Checkpoint:</b>\n{ckpt_name}"
            self.send_message(chat_id, msg)
        elif cmd == "/gpu":
            gpu_str = ", ".join(self.gpu_devices) if self.gpu_devices else "CPU"
            vram_str = "N/A"
            if torch.cuda.is_available():
                total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
                alloc_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
                vram_str = f"{alloc_gb:.2f} / {total_gb:.2f} GB"
            msg = f"<b>GPU Device:</b> {gpu_str}\n<b>VRAM:</b> {vram_str}"
            self.send_message(chat_id, msg)
        elif cmd == "/ping":
            self.send_message(chat_id, "GrainSpeech training supervisor is active.")

    def run_poller(self):
        while not self.stop_event.is_set():
            self.fetch_updates()
            time.sleep(2)

    def start(self):
        t = threading.Thread(target=self.run_poller, daemon=True)
        t.start()
        gpu_info = ", ".join(self.gpu_devices) if self.gpu_devices else "CPU"
        self.fetch_updates()
        self.broadcast(
            "<b>GrainSpeech Training Launched</b>\n"
            f"Device: {gpu_info}\n"
            f"Batch Size: {BATCH_SIZE}\n"
            f"Precision: {OPTIMAL_PRECISION}\n"
            f"Status: Initializing supervisor..."
        )

    def stop(self):
        self.stop_event.set()

monitor = TelegramTrainingMonitor(TELEGRAM_BOT_TOKEN)
monitor.start()

def parse_line_for_metrics(line):
    ep_match = re.search(r"Epoch\s+(\d+)", line)
    st_match = re.search(r"Step\s+(\d+)(?:/(\d+))?", line)
    loss_match = re.search(r"Loss:\s*([0-9\.]+)", line)
    pct_match = re.search(r"(\d+\.?\d*)\s*%", line)
    if ep_match:
        monitor.current_epoch = ep_match.group(1)
    if st_match:
        monitor.current_step = st_match.group(1)
    if loss_match:
        monitor.current_loss = loss_match.group(1)
    if pct_match:
        monitor.progress_pct = pct_match.group(0)

    ep_pbar = re.search(r"Epoch\s+(\d+):\s*(\d+)%", line)
    if ep_pbar:
        monitor.current_epoch = ep_pbar.group(1)
        monitor.progress_pct = f"{ep_pbar.group(2)}%"

    loss_pbar = re.search(r"loss=([0-9\.]+)", line)
    if loss_pbar:
        monitor.current_loss = loss_pbar.group(1)

_port_counter = 0

def _get_free_port():
    global _port_counter
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(("", 0))
        port = s.getsockname()[1]
        s.close()
        return str(port)
    except Exception:
        _port_counter += 1
        return str(29500 + (_port_counter % 500))

def run_training_supervised(max_retries=10):
    consecutive_failures = 0
    total_epochs = 5000
    last_housekeeping_time = time.time()

    while consecutive_failures < max_retries:
        resume_ckpt = find_best_resume_checkpoint()
        monitor.active_checkpoint = resume_ckpt
        if resume_ckpt:
            print(f"[GrainSpeech] Resuming training from checkpoint: {resume_ckpt}")
        else:
            print("[GrainSpeech] Starting fresh training (no prior checkpoint).")

        os.environ["GRAINSPEECH_CHECKPOINT_DIR"] = LOCAL_CHECKPOINTS
        os.environ["GRAINSPEECH_EVERY_N_EPOCHS"] = "1"
        os.environ["GRAINSPEECH_SAVE_TOP_K"] = "3"
        os.environ["PYTHONPATH"] = f"{KAGGLE_WORKING}{os.pathsep}{LOCAL_REPO}{os.pathsep}{grainspeech_pkg}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"
        os.environ["MALLOC_ARENA_MAX"] = "2"
        os.environ["MALLOC_MMAP_THRESHOLD_"] = "131072"
        os.environ["MALLOC_TRIM_THRESHOLD_"] = "131072"
        os.environ["PYTHONMALLOC"] = "malloc"
        os.environ["TORCH_NCCL_HEARTBEAT_TIMEOUT_SEC"] = "3600"
        os.environ["NCCL_TIMEOUT"] = "3600"
        os.environ["NCCL_ASYNC_ERROR_HANDLING"] = "1"
        os.environ["TORCH_DISTRIBUTED_DEBUG"] = "OFF"
        os.environ["NCCL_DEBUG"] = "WARN"

        num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
        workers_per_gpu = 2 if num_gpus <= 2 else 1

        cmd = [
            sys.executable, "-u", "grainspeech/train_l1_ssim_gvar.py",
            "--preprocess-config", config_yaml_path,
            "--run-name", EXPERIMENT_NAME,
            "--batch-size", str(BATCH_SIZE),
            "--num_workers", str(workers_per_gpu),
            "--lr", "0.001",
            "--weight-decay", "0.00001",
            "--max_epochs", str(total_epochs),
            "--precision", OPTIMAL_PRECISION,
            "--out-folder", VAL_OUTPUTS,
            "--hifigan-checkpoint", HIFIGAN_CKPT,
        ]

        if resume_ckpt and os.path.isfile(resume_ckpt):
            cmd.extend(["--checkpoint", resume_ckpt])

        print(f"[GrainSpeech] Launching command: {' '.join(cmd)}")

        proc_env = os.environ.copy()
        ddp_port = _get_free_port()
        proc_env["MASTER_PORT"] = ddp_port
        proc_env["MAIN_PORT"] = ddp_port
        print(f"[GrainSpeech] Assigned DDP Master Port: {ddp_port}")

        subprocess.run("pkill -9 -f 'train_l1_ssim_gvar' ; pkill -9 -f 'grainspeech' ; pkill -9 -f 'torch.distributed'", shell=True, check=False)
        time.sleep(1)

        proc = subprocess.Popen(
            cmd,
            cwd=LOCAL_REPO,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=proc_env
        )
        monitor.training_process = proc

        last_sync_time = time.time()
        error_lines = []
        last_progress_str = ""

        try:
            for line in iter(proc.stdout.readline, ""):
                if not line:
                    break

                parse_line_for_metrics(line)

                is_progress_line = ("Epoch" in line and "%" in line) or ("loss=" in line and "v_num=" in line)
                is_alert_line = any(w in line for w in ("Error", "Exception", "Traceback", "OOM", "CUDA error", "ALERT"))
                is_cpp_noise = any(w in line for w in ("c10::Error", "ProcessGroupNCCL", "libc10.so", "TCPStore", "recvBytes", "sendBytes", f"frame {chr(35)}"))

                if is_progress_line:
                    clean_line = line.strip()
                    if clean_line != last_progress_str:
                        sys.stdout.write(f"\rPROGRESS: {clean_line}")
                        sys.stdout.flush()
                        last_progress_str = clean_line
                elif is_alert_line and not is_cpp_noise:
                    print(f"\n[ALERT] {line.strip()}")
                elif not is_cpp_noise:
                    print(line, end="")

                if any(err_kw in line for err_kw in ("Error", "Exception", "Traceback", "OOM", "CUDA error")) and not is_cpp_noise:
                    error_lines.append(line.strip())
                    if len(error_lines) > 15:
                        error_lines.pop(0)

                now = time.time()
                if now - last_sync_time >= 30:
                    check_and_sync_new_checkpoints()
                    last_sync_time = now

                if now - last_housekeeping_time >= 1800:
                    run_housekeeping(keep_local=3, keep_hf=5)
                    last_housekeeping_time = now

            proc.stdout.close()
            ret_code = proc.wait()

        except KeyboardInterrupt:
            print("\n[GrainSpeech] Training interrupted by user.")
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
            monitor.broadcast("<b>GrainSpeech Training Halted by User</b>")
            break

        check_and_sync_new_checkpoints()

        best_after = find_best_resume_checkpoint()
        if best_after:
            print(f"[GrainSpeech] Final checkpoint sync to Hugging Face: {best_after}")
            sync_checkpoint_to_hf(best_after)

        if ret_code == 0:
            print("[GrainSpeech] Training completed successfully!")
            monitor.broadcast("<b>GrainSpeech Training Completed Successfully!</b>")
            break
        else:
            consecutive_failures += 1
            print(f"\n[ERROR] Process terminated with exit code {ret_code}.")
            if error_lines:
                print("Recent output lines:")
                for el in error_lines[-10:]:
                    print(el)

            monitor.broadcast(
                f"<b>Training Interrupted (Exit {ret_code})</b>\n"
                f"Epoch: {monitor.current_epoch}, Step: {monitor.current_step}\n"
                f"Loss: {monitor.current_loss}\n"
                f"Auto-resuming attempt {consecutive_failures}/{max_retries}..."
            )

            time.sleep(5)
            run_housekeeping(keep_local=3, keep_hf=5)
            subprocess.run("pkill -9 -f 'train_l1_ssim_gvar' ; pkill -9 -f 'grainspeech' ; pkill -9 -f 'torch.distributed'", shell=True, check=False)
            time.sleep(2)
            print(f"[GrainSpeech] Auto-resuming training in 5 seconds (attempt {consecutive_failures}/{max_retries})...")

    monitor.stop()

run_training_supervised(max_retries=10)


In [ ]:
import shutil
import subprocess
import glob
import sys
import os
QUANTIZE_INT8 = False
TEST_TEXT = "مرحبا بكم في تجربة نموذج جرين سبيتش لتحويل النص الى كلام عالي الجودة"

all_c = find_local_ckpts() if "find_local_ckpts" in dir() else glob.glob(os.path.join(LOCAL_CHECKPOINTS, "*.ckpt"))
checkpoint_path = all_c[-1] if all_c else None

if checkpoint_path and os.path.exists(checkpoint_path):
    print(f"Loading checkpoint for inference & export: {checkpoint_path}")
    from model_l1_ssim_gvar import GrainSpeech, get_hifigan
    from text import text_to_sequence
    import yaml
    from IPython.display import Audio, display

    with open(config_yaml_path, "r", encoding="utf-8") as f:
        preprocess_config = yaml.safe_load(f)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    try:
        model = GrainSpeech.load_from_checkpoint(
            checkpoint_path,
            preprocess_config=preprocess_config,
            hifigan_checkpoint=HIFIGAN_CKPT,
            infer_device=device,
            map_location=device,
            weights_only=False,
        )
    except Exception:
        model = GrainSpeech(
            preprocess_config=preprocess_config,
            hifigan_checkpoint=HIFIGAN_CKPT,
            infer_device=device,
        )
        ckpt_data = torch.load(checkpoint_path, map_location=device, weights_only=False)
        st = ckpt_data.get("state_dict", ckpt_data)
        model.load_state_dict(st, strict=False)

    model.eval().to(device)
    vocoder = get_hifigan(checkpoint=HIFIGAN_CKPT, infer_device=device)

    cleaners = preprocess_config["preprocessing"]["text"]["text_cleaners"]
    seq = text_to_sequence(TEST_TEXT, cleaners)
    if seq:
        in_tensor = torch.tensor([seq], dtype=torch.long, device=device)
        with torch.no_grad():
            pred = model.phoneme2mel(in_tensor)
            mel = pred[1] if isinstance(pred, (list, tuple)) else (pred["mel"] if isinstance(pred, dict) else pred)
            if mel.dim() == 3:
                mel = mel.transpose(1, 2)
            if vocoder is not None:
                wav = vocoder(mel).squeeze().cpu().numpy()
                display(Audio(wav, rate=SAMPLE_RATE))

    class GrainSpeechOnnxExport(torch.nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m.phoneme2mel

        def forward(self, phonemes):
            out = self.m(phonemes)
            mel = out[1] if isinstance(out, (list, tuple)) else (out["mel"] if isinstance(out, dict) else out)
            return mel

    onnx_file = os.path.join(LOCAL_ONNX_EXPORT, "grainspeech_kawthar.onnx")
    wrapper = GrainSpeechOnnxExport(model)
    dummy_input = torch.randint(1, 80, (1, 30), dtype=torch.long, device=device)

    try:
        torch.onnx.export(
            wrapper,
            (dummy_input,),
            onnx_file,
            input_names=["phonemes"],
            output_names=["mel"],
            dynamic_axes={"phonemes": {1: "seq_len"}, "mel": {1: "time_frames"}},
            opset_version=14,
        )
        old_onnx = hf_list_files(HF_ONNX_PREFIX)
        if old_onnx:
            hf_delete_files(old_onnx)
        hf_upload_folder(LOCAL_ONNX_EXPORT, HF_ONNX_PREFIX)
        print("Cell 8 Complete: ONNX models exported & uploaded to Hugging Face successfully.")
    except Exception as e:
        print(f"ONNX export notice: {e}")
else:
    print("No checkpoint found for ONNX export.")
